# Part  3: Modeling

In [1]:
import joblib 
import numpy as np

X_train_scaled = joblib.load('../data/processed/X_train_scaled.pkl')
X_test_scaled = joblib.load('../data/processed/X_test_scaled.pkl')

y_train = joblib.load('../data/processed/y_train.pkl')  
y_test = joblib.load('../data/processed/y_test.pkl')

print(X_train_scaled.shape)
print(X_test_scaled.shape)
print(y_train.shape)
print(y_test.shape)

(1166, 229)
(292, 229)
(1166,)
(292,)


***3.1 Linear Regression***

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np


lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train) # learn by OLS - thuật toán bình phương tối thiểu 
y_pred_lr = lr_model.predict(X_test_scaled)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr) #r2 score

print("Linear Regression:") 
print(f"RMSE: {rmse_lr:.2f}") 
print(f"MAE: {mae_lr:.2f}")
print(f"R²: {r2_lr:.4f}")

Linear Regression:
RMSE: 27888.79
MAE: 19845.59
R²: 0.8592


**⇒ Linear regression predicts quite accurately.**

It is trained using **Ordinary Least Squares (OLS)** because the dataset is relatively small. OLS finds the best-fitting line by minimizing the sum of squared residuals. 

For larger datasets, **Gradient Descent** is often preferred because it is more computationally efficient than solving the OLS optimization directly.


***3.2 Random forest***

In [8]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)#n_estiomators: the number of decision trees
rf_model.fit(X_train_scaled, y_train) 

y_pred_rf = rf_model.predict(X_test_scaled)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print('RandomForest:')
print(f"RMSE: {rmse_rf:.2f}")
print(f"MAE: {mae_rf:.2f}")
print(f"R2 {r2_rf: 4f}")


RandomForest:
RMSE: 24587.38
MAE: 17156.78
R2  0.890556


=> As can be seen, **Random Forest outperforms Linear Regression** based on the **RMSE**, **MAE**, and **R²**


***3.3 XGBoost-eXtreme Gradient Boosting***

In [11]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators = 100, random_state = 42)
xgb_model.fit(X_train_scaled, y_train)

y_pred_xgb = xgb_model.predict(X_test_scaled)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost:")
print(f"RMSE: {rmse_xgb:.2f}")
print(f"MAE: {mae_xgb:.2f}")
print(f"R²: {r2_xgb:.4f}")

XGBoost:
RMSE: 23338.81
MAE: 16117.31
R²: 0.9014


As can be seen, **XGBoost outperforms Random Forest** based on the **RMSE**, **MAE**, and **R²** metrics.

Therefore, **XGBoost is selected as the final model** for prediction.


In [15]:
X_train = joblib.load('../data/processed/X_train.pkl')

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15))

                  feature  importance
4             OverallQual    0.408151
33             GarageCars    0.091873
201     GarageType_Detchd    0.064737
21              GrLivArea    0.035853
29           TotRmsAbvGrd    0.033880
16            TotalBsmtSF    0.033464
28            KitchenQual    0.031786
186          CentralAir_Y    0.026437
11               BsmtQual    0.021218
31            FireplaceQu    0.020522
36             GarageCond    0.016059
72   Neighborhood_Crawfor    0.015186
13             BsmtFinSF1    0.014402
197     GarageType_Attchd    0.012708
130   Exterior1st_BrkFace    0.010747


Conclusion — Correlation (EDA) vs Feature Importance (XGBoost)

<i>
OverallQualverallQual-correlation cao nhất (0.79) và importance (0.41, gấp 4-5 lần feature #2) → feature chốt của model

GarageCars giữ vị trí mạnh ở cả 2 (corr 0.64 / importance 0.09, hạng 2-3)

GrLivArea tụt hạng rõ rệt: correlation cao thứ 2 (0.71) nhưng importance chỉ hạng 4 (0.036) → do multicollinearity: GrLivArea tương quan với TotalBsmtSF, 1stFlrSF (đều đo diện tích), khi model đã dùng các feature diện tích khác thì GrLivArea không còn thêm nhiều thông tin mới

Feature ẩn correlation không thấy cần XGBoost:
GarageType_Detchd, CentralAir_Y — là cột one-hot từ biến chữ, không nằm trong bảng correlation gốc -> chứng minh correlation tuyến tính bỏ sót thông tin từ biến categorical
<i>

***GridSearchCV***

In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],#default is 6
    'learning_rate': [0.05, 0.1, 0.2]#default is 0.1
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best RMSE:", np.sqrt(-grid_search.best_score_))

Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}
Best RMSE: 24031.202916208753


In [20]:
best_xgb_model = grid_search.best_estimator_

y_pred_best = best_xgb_model.predict(X_test_scaled)

rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
mae_best = mean_absolute_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

print("Best XGBoost:")
print(f"RMSE: {rmse_best:.2f}")
print(f"MAE: {mae_best:.2f}")
print(f"R²: {r2_best:.4f}")

Best XGBoost:
RMSE: 20232.65
MAE: 14772.80
R²: 0.9259


In [ ]:
import joblib

joblib.dump(best_xgb_model, '../model/house_price_model.pkl')

print("File saved successfully.")

File saved successfully


In [33]:
#Check

loaded_model = joblib.load('../model/house_price_model.pkl')
loaded_scaler = joblib.load('../model/scaler.pkl')
loaded_columns = joblib.load('../model/feature_columns.pkl')

print("Số lượng feature columns:", len(loaded_columns))

test_sample = X_test_scaled[0].reshape(1, -1)
prediction = loaded_model.predict(test_sample)
performance = (1 - abs(prediction[0] - y_test.iloc[0]) / y_test.iloc[0]) * 100

print("Predictive Price:", prediction[0])
print("Actual Price:", y_test.iloc[0])
print(f"The performance for the first house in dataset: {performance:.2f}%")

Số lượng feature columns: 229
Predictive Price: 212155.12
Actual Price: 190000
The performance for the first house in dataset: 88.34%


## Kết luận — Modeling (Bước 1.3)
<b>

1. So sánh 3 model trên test set: Linear Regression (R² 0.859) < Random Forest (R² 0.891) < XGBoost mặc định (R² 0.901)

2. GridSearchCV fit XGBoost → best params: learning_rate=0.05, max_depth=3, n_estimators=300

3. Final model: XGBoost was fixed — RMSE 20,232.65 / MAE 14,772.80 / R² 0.9259

4. Feature importance khớp phần lớn với correlation (OverallQual), nhưng GrLivArea tụt hạng do multicollinearity; các cột one-hot (GarageType_Detchd, CentralAir_Y) chỉ XGBoost phát hiện được

5. Đã lưu house_price_model.pkl (kèm scaler.pkl, feature_columns.pkl có sẵn) → đủ cho Phase 2
<b>
